# Week 09 — Neural Network Surrogate Training

Same architecture family as W4-W6: MLP with H ∈ {16, 32}, four regularisation variants (plain / dropout / weight-decay / ensemble), 5-fold CV across 8 configs.

Re-trains on the latest data including W6 portal returns (16/16/21/36/26/26/36/46 pts).

F2/F4/F5/F6/F8 had new bests in W6 — fresh NN gradients at these new best points.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
sys.path.append('../src')
import nn_models as nm

MODELS_DIR = Path('../models/week_09')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

WIDTHS = [16, 32]
VARIANTS = list(nm.VARIANTS)

def load(n):
    X = np.load(f'../data/function_{n}/initial_inputs.npy')
    Y = np.load(f'../data/function_{n}/initial_outputs.npy')
    return X, Y

In [2]:
def search_and_save(n, verbose=True):
    X, Y = load(n)
    baseline = float(Y.std())
    results = []
    for H in WIDTHS:
        for v in VARIANTS:
            rmse = nm.cv_rmse(X, Y, v, hidden=H, n_splits=5, seed=0)
            results.append((rmse, H, v))
    results.sort(key=lambda r: r[0])
    best_rmse, best_H, best_v = results[0]
    improvement = (baseline - best_rmse) / baseline * 100
    beat_baseline = best_rmse < baseline

    if verbose:
        print(f'F{n}: {X.shape[0]} pts, {X.shape[1]}D, baseline RMSE = {baseline:.4f}')
        print(f'  All configs (sorted):')
        for r, H, v in results:
            mark = ' ← BEST' if (r, H, v) == (best_rmse, best_H, best_v) else ''
            print(f'    H={H:3d}  {v:10s}  CV RMSE = {r:.4f}{mark}')
        status = '✓ beats baseline' if beat_baseline else '✗ WORSE than baseline'
        print(f'  → best: H={best_H}, {best_v}, RMSE={best_rmse:.4f} ({improvement:+.1f}% vs baseline) {status}')

    # Fit final model on all data
    models, meta = nm.fit_final(X, Y, best_v, best_H)
    meta['cv_rmse'] = best_rmse
    meta['baseline_rmse'] = baseline
    meta['beats_baseline'] = beat_baseline
    meta['all_configs'] = [{'hidden': H, 'variant': v, 'rmse': r} for r, H, v in results]

    # Gradient at current best point
    x_best = X[Y.argmax()]
    grad = nm.gradient_at(models, meta, x_best)
    meta['gradient_at_best'] = grad.tolist()
    meta['x_best'] = x_best.tolist()
    meta['y_best'] = float(Y.max())

    if verbose:
        print(f'  Gradient dY/dx at best point: {np.round(grad, 3).tolist()}')

    nm.save(models, meta, MODELS_DIR / f'function_{n}_nn.pt')
    return meta

## Train all 8 functions

In [3]:
all_meta = {}
for n in range(1, 9):
    print('=' * 72)
    all_meta[n] = search_and_save(n, verbose=True)
    print()

F1: 18 pts, 2D, baseline RMSE = 0.0017
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.0028 ← BEST
    H= 32  wd          CV RMSE = 0.0032
    H= 16  dropout     CV RMSE = 0.0032
    H= 32  dropout     CV RMSE = 0.0032
    H= 32  plain       CV RMSE = 0.0033
    H= 16  ensemble    CV RMSE = 0.0042
    H= 16  wd          CV RMSE = 0.0048
    H= 16  plain       CV RMSE = 0.0051
  → best: H=32, ensemble, RMSE=0.0028 (-64.7% vs baseline) ✗ WORSE than baseline


  Gradient dY/dx at best point: [0.024000000208616257, -0.004999999888241291]



F2: 18 pts, 2D, baseline RMSE = 0.2393
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.2040 ← BEST
    H= 32  dropout     CV RMSE = 0.2189
    H= 32  wd          CV RMSE = 0.2957
    H= 16  wd          CV RMSE = 0.2957
    H= 32  plain       CV RMSE = 0.2965
    H= 16  plain       CV RMSE = 0.2973
    H= 32  ensemble    CV RMSE = 0.2981
    H= 16  ensemble    CV RMSE = 0.3122
  → best: H=16, dropout, RMSE=0.2040 (+14.7% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [0.7630000114440918, 1.2899999618530273]



F3: 23 pts, 3D, baseline RMSE = 0.0735
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.0827 ← BEST
    H= 32  dropout     CV RMSE = 0.0862
    H= 16  dropout     CV RMSE = 0.0882
    H= 16  ensemble    CV RMSE = 0.0905
    H= 32  plain       CV RMSE = 0.0954
    H= 32  wd          CV RMSE = 0.0955
    H= 16  wd          CV RMSE = 0.0980
    H= 16  plain       CV RMSE = 0.1023
  → best: H=32, ensemble, RMSE=0.0827 (-12.6% vs baseline) ✗ WORSE than baseline


  Gradient dY/dx at best point: [0.11599999666213989, -0.057999998331069946, -0.6190000176429749]



...

F6: 28 pts, 5D, baseline RMSE = 0.6612
  All configs (sorted):
    H= 16  ensemble    CV RMSE = 0.3445 ← BEST
    H= 32  ensemble    CV RMSE = 0.3529
    H= 16  plain       CV RMSE = 0.3563
    H= 16  wd          CV RMSE = 0.3591
    H= 32  wd          CV RMSE = 0.3899
    H= 32  plain       CV RMSE = 0.4000
    H= 16  dropout     CV RMSE = 0.4128
    H= 32  dropout     CV RMSE = 0.4156
  → best: H=16, ensemble, RMSE=0.3445 (+47.9% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-0.2150000035762787, -0.4189999997615814, -1.2899999618530273, -1.2070000171661377, -1.9809999465942383]



F7: 38 pts, 6D, baseline RMSE = 0.5928
  All configs (sorted):
    H= 32  dropout     CV RMSE = 0.4831 ← BEST
    H= 16  dropout     CV RMSE = 0.5155
    H= 32  wd          CV RMSE = 0.6104
    H= 16  wd          CV RMSE = 0.6382
    H= 16  ensemble    CV RMSE = 0.6429
    H= 32  plain       CV RMSE = 0.6660
    H= 32  ensemble    CV RMSE = 0.6755
    H= 16  plain       CV RMSE = 0.6821
  → best: H=32, dropout, RMSE=0.4831 (+18.5% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-0.6570000052452087, -0.3330000042915344, 0.4339999854564667, -0.5590000152587891, -0.5090000033378601, 0.31200000643730164]



F8: 48 pts, 8D, baseline RMSE = 1.1496
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.4444 ← BEST
    H= 32  ensemble    CV RMSE = 0.4809
    H= 32  dropout     CV RMSE = 0.4903
    H= 16  ensemble    CV RMSE = 0.5075
    H= 32  wd          CV RMSE = 0.5300
    H= 32  plain       CV RMSE = 0.5336
    H= 16  wd          CV RMSE = 0.5730
    H= 16  plain       CV RMSE = 0.5846
  → best: H=16, dropout, RMSE=0.4444 (+61.3% vs baseline) ✓ beats baseline
  Gradient dY/dx at best point: [-0.5210000276565552, -0.19699999690055847, -0.7540000081062317, -0.1509999930858612, 0.0430000014603138, 0.01600000075995922, -0.4699999988079071, 0.10199999809265137]



## F1 sign classifier

F1 is numerically ~0 for almost all points. Training an NN classifier on sign(y > 0) gives the analyze step a map of "where is the function positive" — useful for Q3/Q6 in the reflection.

In [4]:
X1, Y1 = load(1)
pos_frac = (Y1 > 0).mean()
print(f'F1: {len(Y1)} pts, {(Y1 > 0).sum()} positive, {(Y1 <= 0).sum()} zero-or-negative ({pos_frac:.0%} positive)')

if 0 < pos_frac < 1:
    clf, loo_acc = nm.train_sign_classifier(X1, Y1, hidden=16)
    nm.save_classifier(clf, loo_acc, d_in=X1.shape[1], hidden=16, path=MODELS_DIR / 'function_1_classifier.pt')
    print(f'Sign classifier trained. LOO accuracy = {loo_acc:.2%}')
else:
    print('Classifier skipped (all samples are one class).')

F1: 18 pts, 12 positive, 6 zero-or-negative (67% positive)


Sign classifier trained. LOO accuracy = 77.78%


## Summary

In [5]:
print(f"{'F':>2}  {'H':>3}  {'variant':10s}  {'CV RMSE':>10s}  {'baseline':>10s}  {'improve%':>8s}  {'beats?':>6s}")
print('-' * 62)
for n, m in all_meta.items():
    improve = (m['baseline_rmse'] - m['cv_rmse']) / m['baseline_rmse'] * 100
    mark = '✓' if m['beats_baseline'] else '✗'
    print(f'{n:>2}  {m["hidden"]:>3}  {m["variant"]:10s}  {m["cv_rmse"]:>10.4f}  {m["baseline_rmse"]:>10.4f}  {improve:>+7.1f}%  {mark:>6s}')

 F    H  variant        CV RMSE    baseline  improve%  beats?
--------------------------------------------------------------
 1   32  ensemble        0.0028      0.0017    -64.7%       ✗
 2   16  dropout         0.2040      0.2393    +14.7%       ✓
 3   32  ensemble        0.0827      0.0735    -12.6%       ✗
 4   16  plain           4.3097      9.4442    +54.4%       ✓
 5   32  plain         106.6607   1061.4207    +90.0%       ✓
 6   16  ensemble        0.3445      0.6612    +47.9%       ✓
 7   32  dropout         0.4831      0.5928    +18.5%       ✓
 8   16  dropout         0.4444      1.1496    +61.3%       ✓
